In [8]:
from pydantic import BaseModel
from pydantic.fields import computed_field


class Parent(BaseModel):
    foo: int
    bar: int

    @property
    def total(self) -> int:
        # works for *any* descendant class
        return sum(getattr(self, name) for name in type(self).model_fields)

    @computed_field
    @property
    def total_computed_field(self) -> int:
        # works for *any* descendant class
        return sum(getattr(self, name) for name in type(self).model_fields)


class Child(Parent):
    baz: int


p = Parent(foo=1, bar=2)
c = Child(foo=1, bar=2, baz=3)

print(p.total)  # → 3   (1+2)
print(c.total)  # → 6   (1+2+3)
c.model_dump()

3
6


{'foo': 1, 'bar': 2, 'baz': 3, 'total_computed_field': 6}

In [12]:
from pydantic import BaseModel, Field
from pydantic.fields import computed_field

from ems_prepared.dialogue_state.type_defs import KnownBoolean


class KnownBooleanLeaf(BaseModel):
    """A model that extends KnownBoolean to include additional attributes."""

    rd1_question: str | None = Field(
        default=None,
        title="RD1 Question",
        description="An optional question related to the rd1 symptoms.",
    )

    test: str
    test2: KnownBoolean
    test3: KnownBoolean

    @computed_field
    @property
    def rd2_symptoms(self) -> set[KnownBoolean]:
        """Returns the set of fields of type KnownBoolean for rd2."""
        output = set()
        for name, field in type(self).model_fields.items():
            print(field.annotation)
            if field.annotation is KnownBoolean:
                output.add(getattr(self, name))

        return output

    @computed_field
    @property
    def rd2(self) -> KnownBoolean:
        """Returns the count of symptoms in rd2_symptoms."""
        return any(self.rd2_symptoms)

In [13]:
kb = KnownBooleanLeaf(
    rd1_question="Is the patient experiencing symptoms?",
    test="example",
    test2=None,
    test3=False,
)
kb.model_dump()

str | None
<class 'str'>
KnownBoolean
KnownBoolean
str | None
<class 'str'>
KnownBoolean
KnownBoolean


{'rd1_question': 'Is the patient experiencing symptoms?',
 'test': 'example',
 'test2': None,
 'test3': False,
 'rd2_symptoms': {False, None},
 'rd2': False}